In [1]:
from pathlib import Path
import sys

import polars as pl

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from ufc_scraper.utils import read_events


In [2]:
events = read_events().collect()

In [3]:
events.head()

event_link,event_name,event_date,event_location,scraped_at,ingested_at,event_id
str,str,str,str,str,str,str
"""http://www.ufcstats.com/event-…","""UFC Fight Night: Hernandez vs.…","""August 22, 2026""","""Sacramento, California, USA""","""2026-09-03T13:16:53+00:00""","""2026-09-03T13:23:46+00:00""","""a0a69dc9914ef6e1"""


In [4]:
EVENT_COLS = [
    "event_id",
    "event_name",
    "event_date",
    "event_location",
    "scraped_at",
    "ingested_at",
]

In [7]:

def standardize_dataframe(
    df: pl.DataFrame,
    empty_to_null: bool = False,
) -> pl.DataFrame:
    """
    Standardize a Polars DataFrame.

    - Strips whitespace from column names.
    - Strips whitespace from all string values.
    - Removes surrounding double quotes from string values.
    - Optionally converts empty strings to nulls.
    """

    # Clean column names
    df = df.rename(
        {
            col: col.strip()
            for col in df.columns
        }
    )

    string_cols = [
        col
        for col, dtype in df.schema.items()
        if dtype == pl.String
    ]

    exprs = []

    for col in string_cols:
        expr = (
            pl.col(col)
            .str.strip_chars()
            .str.strip_chars('"')
        )

        if empty_to_null:
            expr = (
                pl.when(expr == "")
                .then(None)
                .otherwise(expr)
            )

        exprs.append(expr.alias(col))

    return df.with_columns(*exprs)

In [8]:
import polars as pl


EVENT_COLS = [
    "event_id",
    "event_name",
    "event_date",
    "event_location",
    "scraped_at",
    "ingested_at",
]


def transform_events(
    events: pl.DataFrame,
) -> pl.DataFrame:

    return (
        events
        .with_columns(

            # Event date
            pl.col("event_date")
            .str.to_date(
                format="%B %d, %Y"
            ),

            # Datetime columns
            pl.col("scraped_at")
            .str.to_datetime(
                format="%Y-%m-%dT%H:%M:%S%z"
            ),

            pl.col("ingested_at")
            .str.to_datetime(
                format="%Y-%m-%dT%H:%M:%S%z"
            ),

            # Clean text
            pl.col("event_id")
            .str.strip_chars(),

            pl.col("event_name")
            .str.strip_chars(),

            pl.col("event_location")
            .str.strip_chars(),
        )
        .select(EVENT_COLS)
    )


In [9]:
events_clean = (
    events
    .pipe(
        standardize_dataframe,
        empty_to_null=True,
    )
    .pipe(transform_events)
)

In [10]:
events_clean

event_id,event_name,event_date,event_location,scraped_at,ingested_at
str,str,date,str,"datetime[μs, UTC]","datetime[μs, UTC]"
"""a0a69dc9914ef6e1""","""UFC Fight Night: Hernandez vs.…",2026-08-22,"""Sacramento, California, USA""",2026-09-03 13:16:53 UTC,2026-09-03 13:23:46 UTC
